# PyTorch: tensors, autograd, GPU

**M0 · Toolbox & Submission Craft** — [AI Olympiad Curriculum](https://ai-olympiad-curriculum.netlify.app)

~35 min

## Tensors and gradients

A **tensor** is a NumPy array that can also live on a **GPU** (the graphics card — thousands of small cores that multiply matrices far faster than the CPU; PyTorch calls it `"cuda"`). Why PyTorch and not NumPy? Because training a network means adjusting millions of numbers to make a **loss** — a number measuring how wrong the model's output is — smaller, and for that you need the **gradient**: for each number, how much the loss changes if you nudge that number up a little. PyTorch computes gradients for you.



In [ ]:
import torch
x = torch.tensor([2.0, 3.0], requires_grad=True)   # "track x: I will ask for gradients later"
y = (x ** 2).sum()      # y = x1² + x2²  = 4 + 9 = 13
y.backward()            # fill in dy/dx for every tracked tensor
x.grad                  # tensor([4., 6.])

Check it by hand: nudge x1 from 2.0 to 2.01 and y goes from 13 to 13.0401 — about 0.04 for a 0.01 nudge, a slope of 4. That slope is `x.grad[0]`. For y = x² the slope is 2x, so x.grad is `2 * x` = `[4, 6]`. For y = 3·x the slope is 3 everywhere.

`requires_grad=True` tells PyTorch to **record** every operation applied to `x`. `backward()` walks that record backwards, multiplying the slopes of each step (that is all the **chain rule** is), and stores the result in `.grad`. Every neural network is trained this way: compute loss, `backward()`, nudge each weight against its gradient.



## Tensor ops: dtype, shape, NumPy bridge

Torch mirrors the NumPy API: `torch.zeros`, `torch.randn`, `torch.arange`, boolean masks, broadcasting — nearly the whole NumPy lesson applies. The differences that matter: dtype defaults to `float32`, devices must match (`cpu` vs `cuda`), and a tensor may carry a gradient.



In [ ]:
import torch
t = torch.tensor([1, 2, 3], dtype=torch.float32)
torch.randn(4, 4)                      # standard normal, float32
t.reshape(3, 1); t.unsqueeze(0)        # (3,1) and (1,3) — like [:, None] / [None, :]
t.reshape(3, 1).squeeze()              # (3,) — drop all size-1 axes

The bridge back to NumPy: `t.numpy()` shares memory with a CPU tensor, `t.cpu().numpy()` copies off the GPU first. `t.item()` extracts a Python scalar from a size-1 tensor — that is how you print a loss. `t.detach()` cuts a tensor out of the gradient graph without touching its values.



In [ ]:
loss_t = torch.tensor(0.42)
loss_t.item()                    # 0.4199999... — a plain Python float
g = torch.randn(2, requires_grad=True)
g.detach().numpy()               # OK — no grad tracking on the NumPy side
# g.numpy()                      # RuntimeError: requires_grad → detach first

- `view` shares memory and needs a compatible layout; `reshape` copies when it has to — use `reshape` unless microseconds matter.
- `torch.tensor(existing_tensor)` copies but drops the gradient link — `t.clone().detach()` says what you mean.
- `(64,)` and `(64, 1)` broadcast to `(64, 64)` exactly like NumPy — squeeze labels before comparing with argmax output.



## Devices, Dataset, DataLoader

Training rarely feeds the whole dataset at once. A **DataLoader** hands out the data in **batches** — here 64 samples at a time — in a shuffled order each **epoch** (one pass over the data), so every step of training sees a different small slice. `TensorDataset` just pairs the inputs `X` with the labels `y` so the loader can yield them together.



In [ ]:
import torch
import torch.nn as nn   # "import torch.nn as nn" alone does NOT bind the name torch
device = "cuda" if torch.cuda.is_available() else "cpu"
model = nn.Linear(2, 2).to(device)         # Linear(in, out): a weighted sum per output —
                                           # inputs × learned weights + bias; weights are
                                           # what training adjusts. .to(device) → GPU

import numpy as np
from torch.utils.data import TensorDataset, DataLoader
y = df["target"].to_numpy()                        # df comes from the pandas lesson
X = df[["age", "income"]]                          # the numeric feature columns — the answer stays out
X_t = torch.tensor(np.asarray(X, dtype="float32"))  # np.asarray unwraps a DataFrame/list to ndarray; float32 is PyTorch's default
y_t = torch.tensor(np.asarray(y))
ds = TensorDataset(X_t, y_t)                # pairs row i of X with label i
dl = DataLoader(ds, batch_size=64, shuffle=True)
for xb, yb in dl:                           # xb: (64, 2), yb: (64,)
    xb, yb = xb.to(device), yb.to(device)   # move each batch to where the model is

Why `float32`? NumPy makes float64 by default, PyTorch models use float32, and mixing them raises a dtype error — convert once, at the boundary.

> **Note.** The model and the data must be on the same device. "Expected all tensors to be on the same device" is the most common runtime error.



## Saving and loading

Save the **state_dict** — an ordered dict of the parameter tensors — not the whole model object. To load, rebuild the architecture first, then `load_state_dict` fills the weights in. `map_location` lets a GPU-trained checkpoint load on a CPU-only machine.



In [ ]:
torch.save(model.state_dict(), "model.pt")           # save
model2 = nn.Linear(2, 2)                              # rebuild the architecture
model2.load_state_dict(torch.load("model.pt", map_location="cpu"))
model2.eval()                                         # weights restored — inference mode

> **Note.** For mid-training checkpoints also save the optimizer state and epoch: `torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "epoch": e}, "ckpt.pt")`. Resuming without the optimizer state starts Adam's momentum buffers empty — a slightly different run.



## Training mode vs inference mode

Two separate switches are flipped when you stop training and start predicting.

- `model.eval()` changes how some **layers behave**: dropout stops dropping, BatchNorm uses its running averages instead of batch statistics (both defined in AI1). `model.train()` switches back.
- `torch.no_grad()` tells **autograd** to stop recording operations. No graph is built, so memory drops and the forward pass is faster. It does not change what the layers compute.



In [ ]:
import torch
model.eval()
with torch.no_grad():
    scores = model(xb)          # (64, n_classes): one score per class per sample
    preds = scores.argmax(1)    # index of the largest score in each row = predicted class

Forgetting `eval()` gives noisy predictions (dropout still active); forgetting `no_grad()` wastes memory and can run out of GPU memory on large test sets.



### ✏️ x has shape (5,). After y = (3 * x).sum() and y.backward(), what is x.grad?

**A.** None
**B.** The number 15
**C.** Five 3s
**D.** Five 1s

<details><summary>Answer</summary>

**C.** Five 3s

dy/dxᵢ = 3 for every element.

</details>

### ✏️ Why use model.eval() and torch.no_grad() at inference?

**A.** eval() moves the model to the GPU for faster inference; no_grad() frees the optimizer state
**B.** eval() freezes the weights so they cannot change; no_grad() makes the forward pass deterministic
**C.** eval() puts dropout/BatchNorm in inference mode; no_grad() turns off gradient tracking
**D.** eval() reloads the saved checkpoint; no_grad() pins the model to the CPU

<details><summary>Answer</summary>

**C.** eval() puts dropout/BatchNorm in inference mode; no_grad() turns off gradient tracking

Two separate things: how layers behave (eval) and whether autograd records operations (no_grad).

</details>

---

## ✅ Self-check tasks

Implement the function in the 'your code' cell, then run the check cell — it runs randomized tests and shows the first mismatch. Run the helper cell below once first.

In [ ]:
# ── task-checking helper cell — run once ──
import copy
import numpy as np
import math, statistics, itertools, collections, functools, heapq, random

try:
    import sys; sys.stdout.reconfigure(encoding="utf-8", errors="replace")
except Exception:
    pass

_L = {"noFn":"❓ Define a function `{e}` in the solution cell and run it","runErr":"❌ Test {i}: {t}: {m}","wrong":"❌ Test {i}: {r}","badRef":"⚠️ Check harness broke (generator/reference): {t}: {m}","inL":"Input","expL":"Expected","gotL":"Got","ok":"✅ Correct! All {n} tests passed."}

def _nb_copy_kw(kw):
    return {k: (v.copy() if isinstance(v, np.ndarray) else copy.deepcopy(v)) for k, v in kw.items()}

def _nb_jsafe(x, depth=0):
    if depth > 6: return "…"
    if isinstance(x, np.ndarray): return _nb_jsafe(x.tolist(), depth + 1)
    if isinstance(x, np.integer): return int(x)
    if isinstance(x, np.floating): return float(x)
    if isinstance(x, np.bool_): return bool(x)
    if isinstance(x, float): return round(x, 6) if math.isfinite(x) else str(x)
    if isinstance(x, (list, tuple)): return [_nb_jsafe(v, depth + 1) for v in x]
    if isinstance(x, dict): return {str(k): _nb_jsafe(v, depth + 1) for k, v in x.items()}
    if isinstance(x, (int, str, bool)) or x is None: return x
    return repr(x)

def _nb_cmp(exp, got, path=""):
    if isinstance(exp, np.ndarray) or isinstance(got, np.ndarray):
        e, g = np.asarray(exp), np.asarray(got)
        if e.shape != g.shape:
            return f"shape {tuple(g.shape)} != expected {tuple(e.shape)}"
        try:
            if np.issubdtype(e.dtype, np.number) or np.issubdtype(g.dtype, np.number):
                ef, gf = e.astype(float), g.astype(float)
                close = np.isclose(gf, ef, atol=1e-6, rtol=1e-5, equal_nan=True)
                if not bool(close.all()):
                    i = tuple(int(v) for v in np.argwhere(~close)[0])
                    return f"{path}first mismatch at index {i}: got {gf[i]:.6g} vs expected {ef[i]:.6g}"
            elif not np.array_equal(g, e):
                return f"{path}values differ"
        except Exception as ex:
            return f"{path}cannot compare: {ex}"
        return None
    if isinstance(exp, (int, float)) and not isinstance(exp, bool) and isinstance(got, (int, float)) and not isinstance(got, bool):
        if not np.isclose(float(exp), float(got), atol=1e-6, rtol=1e-5):
            return f"{path}{got} != expected {exp}"
        return None
    if isinstance(exp, (list, tuple)) and isinstance(got, (list, tuple)):
        if len(exp) != len(got):
            return f"{path}length {len(got)} != expected {len(exp)}"
        for i, (a, b) in enumerate(zip(exp, got)):
            r = _nb_cmp(a, b, path + f"[{i}].")
            if r: return r
        return None
    if isinstance(exp, dict) and isinstance(got, dict):
        if set(exp) != set(got):
            return f"{path}keys differ"
        for k in exp:
            r = _nb_cmp(exp[k], got[k], path + f"[{k!r}].")
            if r: return r
        return None
    if exp != got:
        return f"{path}{got!r} != expected {exp!r}"
    return None

def _check_task(g, ref_src, entry, n=20, seed=0):
    """g — globals() ноутбука; ref_src — gen() + эталонное решение."""
    fn = g.get(entry)
    if not callable(fn):
        print(_L["noFn"].format(e=entry)); return
    ns = {"np": np, "math": math, "statistics": statistics, "itertools": itertools,
          "collections": collections, "functools": functools, "heapq": heapq, "random": random}
    try:
        exec(ref_src, ns)
        gen, ref = ns["gen"], ns[entry]
    except Exception as e:
        print(_L["badRef"].format(t=type(e).__name__, m=e)); return
    rng = np.random.default_rng(seed)
    for i in range(n):
        try:
            kw = gen(rng)
        except Exception as e:
            print(_L["badRef"].format(t=type(e).__name__, m=e)); return
        try:
            got = fn(**_nb_copy_kw(kw))
        except Exception as e:
            print(_L["runErr"].format(i=i + 1, t=type(e).__name__, m=e)
                  + "\n" + _L["inL"] + ": " + repr(_nb_jsafe(kw))); return
        exp = ref(**_nb_copy_kw(kw))
        r = _nb_cmp(exp, got)
        if r:
            print(_L["wrong"].format(i=i + 1, r=r)
                  + "\n" + _L["inL"] + ": " + repr(_nb_jsafe(kw))
                  + "\n" + _L["expL"] + ": " + repr(_nb_jsafe(exp))
                  + "\n" + _L["gotL"] + ": " + repr(_nb_jsafe(got))); return
    print(_L["ok"].format(n=n))

### Task 1 — One-hidden-layer MLP forward

●●○ · ~10 min

> A one-hidden-layer MLP is two affine maps with a nonlinearity in between: H = ReLU(X @ W1 + b1), out = H @ W2 + b2. Shapes: X is (n, d), W1 is (d, h), W2 is (h, k) — the hidden size h is whatever the matrices say it is.

Implement `mlp_forward(X, W1, b1, W2, b2)`: compute the forward pass `ReLU(X @ W1 + b1) @ W2 + b2` and return the output logits (no softmax on the output).

<details><summary>Hint</summary>

H = np.maximum(X @ W1 + b1, 0); return H @ W2 + b2.

</details>


In [ ]:
import numpy as np

def mlp_forward(X, W1, b1, W2, b2):
    # your code — two layers with a ReLU between
    return X @ W1 + b1  # wrong — missing ReLU and the second layer

In [ ]:
# self-check — run after solving the cell above
_check_task(globals(), "def gen(rng):\n    n, d = int(rng.integers(2, 9)), int(rng.integers(2, 7))\n    h, k = int(rng.integers(2, 7)), int(rng.integers(2, 5))\n    return {\"X\": rng.normal(0, 1, (n, d)),\n            \"W1\": rng.normal(0, 1, (d, h)), \"b1\": rng.normal(0, 1, h),\n            \"W2\": rng.normal(0, 1, (h, k)), \"b2\": rng.normal(0, 1, k)}\n\ndef mlp_forward(X, W1, b1, W2, b2):\n    H = np.maximum(np.asarray(X) @ np.asarray(W1) + np.asarray(b1), 0.0)\n    return H @ np.asarray(W2) + np.asarray(b2)", "mlp_forward", 20)

---

**Go deeper**
- [PyTorch: Learn the Basics](https://pytorch.org/tutorials/beginner/basics/intro.html)
- [Dive into Deep Learning (d2l.ai)](https://d2l.ai/)